<a href="https://colab.research.google.com/github/Hkd225/Cuaca-IDN/blob/main/Cuaca_IDN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import kagglehub

print("Sedang mengunduh dan membaca data...")
path = kagglehub.dataset_download("greegtitan/indonesia-climate")

df_climate = pd.read_csv(os.path.join(path, 'climate_data.csv'))
df_province = pd.read_csv(os.path.join(path, 'province_detail.csv'))
df_station = pd.read_csv(os.path.join(path, 'station_detail.csv'))

df_station_prov = pd.merge(df_station, df_province, on='province_id', how='left')
if 'station_id' not in df_station.columns and 'id' in df_station.columns:
    df_master = pd.merge(df_climate, df_station_prov, left_on='station_id', right_on='id', how='left')
else:
    df_master = pd.merge(df_climate, df_station_prov, on='station_id', how='left')

print("\n--- Penentuan Target Cuaca ---")
df_master['date'] = pd.to_datetime(df_master['date'], dayfirst=True, errors='coerce')
df = df_master.sort_values(['station_id', 'date']).copy()

def tentukan_cuaca(row):
    if row['RR'] > 0.5:
        return 'Hujan'
    else:
        return 'Cerah'

df['Cuaca_Hari_Ini'] = df.apply(tentukan_cuaca, axis=1)
df['Target_Cuaca_Besok'] = df.groupby('station_id')['Cuaca_Hari_Ini'].shift(-1)

fitur_x = ['Tn', 'Tx', 'Tavg', 'RH_avg', 'ff_x', 'ff_avg']
kolom_penting = fitur_x + ['Target_Cuaca_Besok']
df_clean = df[kolom_penting].copy()

df_clean[fitur_x] = df_clean[fitur_x].transform(lambda x: x.ffill().bfill())
df_clean[fitur_x] = df_clean[fitur_x].fillna(df_clean[fitur_x].median())
df_clean = df_clean.dropna()

print("\n--- Mulai Menerapkan Konsep Klasifikasi Biner ---")

label_encoder = LabelEncoder()
df_clean['Target_Cuaca_Besok_Encoded'] = label_encoder.fit_transform(df_clean['Target_Cuaca_Besok'])

daftar_kategori = label_encoder.classes_

new_df = df_clean.drop(columns=['Target_Cuaca_Besok'])
dataset = new_df.values

X = dataset[:, 0:6]
y = dataset[:, 6]

min_max_scaler = preprocessing.MinMaxScaler()
X_scale = min_max_scaler.fit_transform(X)

X_train, X_test, Y_train, Y_test = train_test_split(X_scale, y, test_size=0.3, shuffle=False)

print("\n--- Membangun dan Melatih Model ---")

model = Sequential([
    Dense(64, activation='relu', input_shape=(6,)),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='Adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

hist = model.fit(X_train, Y_train, epochs=100)

print("\n--- Evaluasi Model pada Data Uji ---")

model.evaluate(X_test, Y_test, batch_size=32)

print("\n--- Simulasi Prediksi Cuaca Untuk Wilayah Baru ---")

def prediksi_cuaca_wilayah(data_input):
    input_array = np.array([[
        data_input['Tn'], data_input['Tx'], data_input['Tavg'],
        data_input['RH_avg'], data_input['ff_x'], data_input['ff_avg']
    ]])

    input_scaled = min_max_scaler.transform(input_array)

    pred_prob = model.predict(input_scaled, verbose=0)[0][0]

    pred_class_idx = 1 if pred_prob > 0.5 else 0

    hasil_prediksi = daftar_kategori[pred_class_idx]
    return hasil_prediksi

data_surabaya = {
    'Tn': 26.0,
    'Tx': 34.0,
    'Tavg': 30.0,
    'RH_avg': 75.0,
    'ff_x': 5.0,
    'ff_avg': 2.0
}

data_jakarta = {
    'Tn': 25.0,
    'Tx': 32.0,
    'Tavg': 28.5,
    'RH_avg': 88.0,
    'ff_x': 4.0,
    'ff_avg': 1.5
}

print(f"Prediksi Cuaca Besok di SURABAYA : {prediksi_cuaca_wilayah(data_surabaya)}")
print(f"Prediksi Cuaca Besok di JAKARTA  : {prediksi_cuaca_wilayah(data_jakarta)}")

Sedang mengunduh dan membaca data...
Using Colab cache for faster access to the 'indonesia-climate' dataset.

--- Penentuan Target Cuaca ---

--- Mulai Menerapkan Konsep Klasifikasi Biner ---

--- Membangun dan Melatih Model ---
Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


12887/12887 ━━━━━━━━━━━━━━━━━━━━ 31s 2ms/step - accuracy: 0.6068 - loss: 0.6570
Epoch 2/100
12887/12887 ━━━━━━━━━━━━━━━━━━━━ 29s 2ms/step - accuracy: 0.6595 - loss: 0.6191
Epoch 3/100
12887/12887 ━━━━━━━━━━━━━━━━━━━━ 30s 2ms/step - accuracy: 0.6640 - loss: 0.6123
Epoch 4/100
12887/12887 ━━━━━━━━━━━━━━━━━━━━ 29s 2ms/step - accuracy: 0.6653 - loss: 0.6096
Epoch 5/100
12887/12887 ━━━━━━━━━━━━━━━━━━━━ 29s 2ms/step - accuracy: 0.6651 - loss: 0.6091
Epoch 6/100
12887/12887 ━━━━━━━━━━━━━━━━━━━━ 29s 2ms/step - accuracy: 0.6656 - loss: 0.6075
Epoch 7/100
12887/12887 ━━━━━━━━━━━━━━━━━━━━ 29s 2ms/step - accuracy: 0.6675 - loss: 0.6060
Epoch 8/100
12887/12887 ━━━━━━━━━━━━━━━━━━━━ 31s 2ms/step - accuracy: 0.6676 - loss: 0.6064
Epoch 9/100
12887/12887 ━━━━━━━━━━━━━━━━━━━━ 45s 3ms/step - accuracy: 0.6693 - loss: 0.6045
Epoch 10/100
12887/12887 ━━━━━━━━━━━━━━━━━━━━ 29s 2ms/step - accuracy: 0.6670 - loss: 0.6058
Epoch 11/100
12887/12887 ━━━━━━━━━━━━━━━━━━━━ 41s 2ms/step - accuracy: 0.6693 - loss: 0.604

In [14]:
import joblib
print("\n--- Menyimpan Model dan Prapemrosesan ---")

nama_file_model = 'model_cuaca_biner.keras'
model.save(nama_file_model)
print(f"Model berhasil disimpan sebagai: {nama_file_model}")

nama_file_scaler = 'scaler_cuaca.pkl'
joblib.dump(min_max_scaler, nama_file_scaler)
print(f"Scaler berhasil disimpan sebagai: {nama_file_scaler}")

nama_file_encoder = 'encoder_cuaca.pkl'
joblib.dump(label_encoder, nama_file_encoder)
print(f"Encoder berhasil disimpan sebagai: {nama_file_encoder}")


--- Menyimpan Model dan Prapemrosesan ---
Model berhasil disimpan sebagai: model_cuaca_biner.keras
Scaler berhasil disimpan sebagai: scaler_cuaca.pkl
Encoder berhasil disimpan sebagai: encoder_cuaca.pkl


In [7]:
print("\n--- Simulasi Prediksi Cuaca Untuk Wilayah Baru ---")

def prediksi_cuaca_wilayah(data_input):
    input_array = np.array([[
        data_input['Tn'], data_input['Tx'], data_input['Tavg'],
        data_input['RH_avg'], data_input['ff_x'], data_input['ff_avg']
    ]])

    input_scaled = min_max_scaler.transform(input_array)

    pred_prob = model.predict(input_scaled, verbose=0)[0][0]

    prediksi_kelas = 1 if pred_prob > 0.5 else 0
    hasil_teks = "HUJAN 🌧️" if prediksi_kelas == 1 else "TIDAK HUJAN / CERAH ☀️"
    peluang_hujan = pred_prob * 100

    nama_kota = data_input.get('nama_kota', 'KOTA TIDAK DIKETAHUI')

    print("======================================================")
    print(f"📍 PREDIKSI CUSTOM UNTUK: {nama_kota.upper()}")
    print("------------------------------------------------------")
    print(f"   Input Suhu      : Min {data_input['Tn']}°C | Max {data_input['Tx']}°C | Rata-rata {data_input['Tavg']}°C")
    print(f"   Input Kelembapan: {data_input['RH_avg']}%")
    print(f"   Input Angin     : {data_input['ff_avg']} m/s (Max {data_input['ff_x']} m/s)")
    print("------------------------------------------------------")
    print(f"   Prediksi Besok  : {hasil_teks}")
    print(f"   Peluang Hujan   : {peluang_hujan:.1f}%")
    print("======================================================\n")

    return hasil_teks

data_surabaya = {
    'nama_kota': 'Surabaya',
    'Tn': 26.0,
    'Tx': 34.0,
    'Tavg': 30.0,
    'RH_avg': 75.0,
    'ff_x': 5.0,
    'ff_avg': 2.0
}

data_jakarta = {
    'nama_kota': 'Jakarta',
    'Tn': 25.0,
    'Tx': 32.0,
    'Tavg': 28.5,
    'RH_avg': 88.0,
    'ff_x': 4.0,
    'ff_avg': 1.5
}

prediksi_cuaca_wilayah(data_surabaya)
prediksi_cuaca_wilayah(data_jakarta)


--- Simulasi Prediksi Cuaca Untuk Wilayah Baru ---
📍 PREDIKSI CUSTOM UNTUK: SURABAYA
------------------------------------------------------
   Input Suhu      : Min 26.0°C | Max 34.0°C | Rata-rata 30.0°C
   Input Kelembapan: 75.0%
   Input Angin     : 2.0 m/s (Max 5.0 m/s)
------------------------------------------------------
   Prediksi Besok  : TIDAK HUJAN / CERAH ☀️
   Peluang Hujan   : 24.0%

📍 PREDIKSI CUSTOM UNTUK: JAKARTA
------------------------------------------------------
   Input Suhu      : Min 25.0°C | Max 32.0°C | Rata-rata 28.5°C
   Input Kelembapan: 88.0%
   Input Angin     : 1.5 m/s (Max 4.0 m/s)
------------------------------------------------------
   Prediksi Besok  : TIDAK HUJAN / CERAH ☀️
   Peluang Hujan   : 49.9%



'TIDAK HUJAN / CERAH ☀️'

In [11]:
print("\n--- Simulasi Prediksi Cuaca Untuk Wilayah Baru ---")

def prediksi_cuaca_wilayah(data_input):
    input_array = np.array([[
        data_input['Tn'], data_input['Tx'], data_input['Tavg'],
        data_input['RH_avg'], data_input['ff_x'], data_input['ff_avg']
    ]])

    input_scaled = min_max_scaler.transform(input_array)

    pred_prob = model.predict(input_scaled, verbose=0)[0][0]

    prediksi_kelas = 1 if pred_prob > 0.5 else 0
    hasil_teks = "HUJAN 🌧️" if prediksi_kelas == 1 else "TIDAK HUJAN / CERAH ☀️"
    peluang_hujan = pred_prob * 100

    nama_kota = data_input.get('nama_kota', 'KOTA TIDAK DIKETAHUI')

    print("======================================================")
    print(f"📍 PREDIKSI CUSTOM UNTUK: {nama_kota.upper()}")
    print("------------------------------------------------------")
    print(f"   Input Suhu      : Min {data_input['Tn']}°C | Max {data_input['Tx']}°C | Rata-rata {data_input['Tavg']}°C")
    print(f"   Input Kelembapan: {data_input['RH_avg']}%")
    print(f"   Input Angin     : {data_input['ff_avg']} m/s (Max {data_input['ff_x']} m/s)")
    print("------------------------------------------------------")
    print(f"   Prediksi Besok  : {hasil_teks}")
    print(f"   Peluang Hujan   : {peluang_hujan:.1f}%")
    print("======================================================\n")

    return hasil_teks

data_surabaya = {
    'nama_kota': 'Surabaya',
    'Tn': 24.0,
    'Tx': 34.0,
    'Tavg': 29.0,
    'RH_avg': 81.5,
    'ff_x': 7.7,
    'ff_avg': 3.0
}

data_jakarta = {
    'nama_kota': 'Jakarta',
    'Tn': 24.0,
    'Tx': 29.0,
    'Tavg': 26.5,
    'RH_avg': 88.0,
    'ff_x': 4.0,
    'ff_avg': 1.5
}

prediksi_cuaca_wilayah(data_surabaya)
prediksi_cuaca_wilayah(data_jakarta)


--- Simulasi Prediksi Cuaca Untuk Wilayah Baru ---
📍 PREDIKSI CUSTOM UNTUK: SURABAYA
------------------------------------------------------
   Input Suhu      : Min 24.0°C | Max 34.0°C | Rata-rata 29.0°C
   Input Kelembapan: 81.5%
   Input Angin     : 3.0 m/s (Max 7.7 m/s)
------------------------------------------------------
   Prediksi Besok  : TIDAK HUJAN / CERAH ☀️
   Peluang Hujan   : 32.2%

📍 PREDIKSI CUSTOM UNTUK: JAKARTA
------------------------------------------------------
   Input Suhu      : Min 24.0°C | Max 29.0°C | Rata-rata 26.5°C
   Input Kelembapan: 88.0%
   Input Angin     : 1.5 m/s (Max 4.0 m/s)
------------------------------------------------------
   Prediksi Besok  : HUJAN 🌧️
   Peluang Hujan   : 59.9%



'HUJAN 🌧️'